1. Load and preprocess data:

• Load the dataset and preprocess the reviews (e.g., lowercasing, re-
moving stop words and punctuation)

In [ ]:
import pandas
import regex as re 
import nltk
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
commentery = pandas.read_csv("train_sentiment.csv")
reviews = commentery["review"].str.lower()
for i in range(len(reviews)) :
  reviews[i]= re.sub(r'[^\w\s]', ' ', reviews[i])
  reviews[i] = re.sub(r'http\S+|www\S+','', reviews[i])
  reviews[i] = re.sub(r'\d+', '', reviews[i])
  reviews[i] = [word for word in word_tokenize(reviews[i]) if ((word not in stopwords.words("english")) and (word not in string.punctuation) and (word.isascii()))]
 #results in the file named"prepro" 
reviews

.Extracing Labels


In [ ]:
import numpy
unprocessed_labels = commentery["rating"].replace(numpy.nan,0).tolist()
unplabel = pandas.DataFrame(unprocessed_labels, columns=["colummn"])
unplabel.to_csv('list.csv', index=False)


In [ ]:
import statistics
gilgilak = []
gilgil = []
for i in range(len(unprocessed_labels)) : 
    try:
        if float(unprocessed_labels[i]) >= 4: 
          gilgilak.append(1)
        else: 
          gilgilak.append(0)
    except :
         gilgilak.append(0)  
labels = pandas.DataFrame(gilgilak, columns=["colummn"])
labels.to_csv('plabel.csv', index=False)
#print(statistics.mean(gilgil[])) 
  

!!!!!!!!!GILGILAK IS THE LABELIZED LISTH WITH TRESHOLD 3!!!!!!!!!



the data set is imbalanced! we have only 10 numbers with treshold lower than 3... this will lead us to an amplified result
so we change it to 4 

2. **Vectorizing reviews**:

In [ ]:
import bert
from sentence_transformers import SentenceTransformer
import gensim
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec


BERT

In [ ]:
bert_model = SentenceTransformer('all-MiniLM-L6-v2')
bert_vectors = bert_model.encode(reviews, convert_to_numpy=True)

TF-IDF


In [ ]:
tfidf_model = TfidfVectorizer(max_features=5000)
#needs modif
reviews = [" ".join(review) if isinstance(review, list) else str(review) for review in reviews]
tfidf_vectors = tfidf_model.fit_transform(reviews)
tfidf_vectors = tfidf_vectors.toarray()


Word2Vec

In [ ]:
w2v_model = Word2Vec(sentences=reviews, vector_size=100, min_count=1)
review_vectors = []
for review in reviews : 
    review_vector = numpy.zeros((100))
    for word in review : 
        if word in w2v_model.wv : 
            review_vector += ((w2v_model.wv[word]) / len(review))
    review_vectors.append(review_vector)

***Testing the performance for logistic regression*** : 

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score , f1_score


In [ ]:
def Logisticregression(model,model_vectors,labels): 
    x_train, x_test, y_train, y_test = train_test_split(model_vectors, labels, test_size=0.2, random_state=42)
    logistic_regression = LogisticRegression()
    logistic_regression.fit(x_train, y_train)
    y_prediction_LOG = logistic_regression.predict(x_test)
    print(f"Detuned Accuracy for {model}:",accuracy_score(y_test, y_prediction_LOG))
    print(f"Detuned F1 Score For {model}:", f1_score(y_test, y_prediction_LOG))
    param_grid_logistic = {
    #ACCORDING TO https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
    'penalty': [None,'l1','l2','elasticnet'],
    'C': [0.001,0.3,5,100],
    'solver': ['lbfgs','liblinear','saga'],
    'max_iter': [20,100],    
    'fit_intercept': [True, False],  
    'class_weight': [None, 'balanced'],
    'tol': [1e-4, 1e-3, 1e-2] 
    }
    grid_log = GridSearchCV(logistic_regression, param_grid_logistic, cv=5, scoring='f1', n_jobs=-1)
    grid_log.fit(x_train, y_train)
    print(f"BEST KNN PARAMETERS FOR{model}: {grid_log.best_params_}")
    print(f"Best Score FOR{model}: {grid_log.best_score_}")
    best_log = grid_log.best_estimator_
    y_pred_log = best_log.predict(x_test)
    print(f"Best KNN F1 Score For {model}: {f1_score(y_test, y_pred_log)}")
    print("-----------------------------------------------------------------------")


In [ ]:
Logisticregression("BERT",bert_vectors,gilgilak)
Logisticregression("Word2Vec",review_vectors,gilgilak)
Logisticregression("TF-IFD",tfidf_vectors,gilgilak)

BEST KNN PARAMETERS FORTF-IFD: {'C': 0.3, 'class_weight': None, 'fit_intercept': True, 'max_iter': 20, 'penalty': None, 'solver': 'saga', 'tol': 0.001}
Best Score FORTF-IFD: 0.8861746485641838
Best KNN F1 Score For TF-IFD: 0.860813704496788

BEST KNN PARAMETERS FORWord2Vec: {'C': 0.001, 'class_weight': None, 'fit_intercept': True, 'max_iter': 20, 'penalty': None, 'solver': 'lbfgs', 'tol': 0.001}
Best Score FORWord2Vec: 0.8637905562761219
Best KNN F1 Score For Word2Vec: 0.85546875

BEST KNN PARAMETERS FORBERT: {'C': 0.3, 'class_weight': None, 'fit_intercept': False, 'max_iter': 20, 'penalty': 'l2', 'solver': 'lbfgs', 'tol': 0.0001}
Best Score FORBERT: 0.8642110320278995
Best KNN F1 Score For BERT: 0.8532289628180039

**KNN:**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
def KNN (Model,Model_vectors,labels) : 
    x_train_knn, x_test_knn, y_train_knn, y_test_knn = train_test_split(Model_vectors, labels, test_size=0.2, random_state=42)
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(x_train_knn, y_train_knn)
    y_prediction_KNN = knn.predict(x_test_knn)
    print(f"Detuned Accuracy for {Model}:",accuracy_score(y_test_knn, y_prediction_KNN))
    print(f"Detuned F1 Score For {Model}:", f1_score(y_test_knn, y_prediction_KNN))
    #According to https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html
    #Parameters 
    param_grid_knn = {
        'n_neighbors': [3, 7, 9 ,10, 11],
        'leaf_size' : [3,4,5,7],
        'metric': ['euclidean', 'manhattan', 'minkowski'],
        #We dont need p really because we have it for euc and manh
        'weights': ['uniform', 'distance'],
        'n_jobs': [None]
    }
    grid_knn = GridSearchCV(knn, param_grid_knn, cv=5, scoring='f1', n_jobs=-1)
    grid_knn.fit(x_train_knn, y_train_knn)
    print(f"BEST KNN PARAMETERS FOR{Model}: {grid_knn.best_params_}")
    print(f"Best Score FOR{Model}: {grid_knn.best_score_}")
    best_knn = grid_knn.best_estimator_
    y_pred_knn = best_knn.predict(x_test_knn)
    print(f"Best KNN F1 Score For {Model}: {f1_score(y_test_knn, y_pred_knn)}")
    print("-----------------------------------------------------------------------")

In [ ]:
KNN("BERT",bert_vectors,gilgilak)
KNN("Word2Vec",review_vectors,gilgilak)
KNN("TF-idf",tfidf_vectors,gilgilak)

Detuned Accuracy for BERT: 0.7303754266211604
Detuned F1 Score For BERT: 0.8384458077709611
BEST KNN PARAMETERS FORBERT: {'leaf_size': 3, 'metric': 'euclidean', 'n_jobs': None, 'n_neighbors': 11, 'weights': 'distance'}
Best Score FORBERT: 0.8678684435864898
Best KNN F1 Score For BERT: 0.8577235772357723
-----------------------------------------------------------------------
Detuned Accuracy for Word2Vec: 0.726962457337884
Detuned F1 Score For Word2Vec: 0.8347107438016529
BEST KNN PARAMETERS FORWord2Vec: {'leaf_size': 3, 'metric': 'euclidean', 'n_jobs': None, 'n_neighbors': 11, 'weights': 'distance'}
Best Score FORWord2Vec: 0.8693081100776034
Best KNN F1 Score For Word2Vec: 0.8577235772357723
-----------------------------------------------------------------------
Detuned Accuracy for TF-idf: 0.7508532423208191
Detuned F1 Score For TF-idf: 0.8388520971302428
BEST KNN PARAMETERS FORTF-idf: {'leaf_size': 3, 'metric': 'euclidean', 'n_jobs': None, 'n_neighbors': 11, 'weights': 'distance'}
Best Score FORTF-idf: 0.8750724197551264
Best KNN F1 Score For TF-idf: 0.8602150537634409
-----------------------------------------------------------------------

**Random Forest**

In [ ]:
from sklearn.ensemble import RandomForestClassifier


In [ ]:
def Randomforest(model,model_vectors,labels): 
    x_train, x_test, y_train, y_test = train_test_split(model_vectors, labels, test_size=0.2, random_state=42)
    randomforest = RandomForestClassifier()
    randomforest.fit(x_train, y_train)
    y_prediction = randomforest.predict(x_test)
    print(f"Detuned Accuracy for {model}:",accuracy_score(y_test, y_prediction))
    print(f"Detuned F1 Score For {model}:", f1_score(y_test, y_prediction))
    param_grid_rf = {'n_estimators' :[60,20,110],
                     #we also had "log_loss"  parameter but because of the limitation of the pc we dont mention it here for tuning
                     'criterion' : ["gini", "entropy"],
                     
                     #NO IDEA FOR TUNING, we prefer that our tree to be totally pure!) 
                     'max_depth' : [10,None,30],
                    'min_samples_split':[2,5],
                    # this one must be 1 for the best tune
                    #'min_weight_fraction_leaf' : [0],
                    'min_samples_leaf': [1, 2, 4],  
                    'max_features': ['sqrt', 'log2', None],
                    'bootstrap': [True, False],   
                    #'class_weight': [None, 'balanced', 'balanced_subsample'] 
                    # there are a dozen of other parameters that we can tune which i really dont know about and the pc cant really handle the calculations!
                    #maybe its better to search with randomCV not GridCV 
        
        }
    grid_Randomforest = GridSearchCV(randomforest, param_grid_rf, cv=5, scoring='f1', n_jobs=-1)
    grid_Randomforest.fit(x_train, y_train)
    print(f"BEST KNN PARAMETERS FOR{model}: {grid_Randomforest.best_params_}")
    print(f"Best Score FOR{model}: {grid_Randomforest.best_score_}")
    best_rf = grid_Randomforest.best_estimator_
    y_pred_knn = best_rf.predict(grid_Randomforest)
    print(f"Best KNN F1 Score For {model}: {f1_score(y_test, y_pred_knn)}")
    print("-----------------------------------------------------------------------")

In [ ]:
Randomforest("BERT",bert_vectors,gilgilak)
Randomforest("Word2Vec",review_vectors,gilgilak)
Randomforest("TF-IDF",tfidf_vectors,gilgilak)

Because of the given threshold 3, most of the ratings are considered as 1 or Positive, so all these algorithms will produce the same result. we can't really determine which one works better but overall BERT must be the best, after that Word2vec and finally TF-IDF.
by changing the threshold to 3.9 the results will be more disputable
but the probles is that we cant always modify our trshold becuase the relation between our parameters will be meaningless. 